# Fase 4 · M00: Índice EDA Final

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 4 — EDA Final |
| **Módulo** | M00 — Índice |

---

## 🎯 Qué hace

Genera la página índice HTML de Fase 4 con la lista de módulos de EDA final y su descripción.

## 📋 Requisitos

- `src/html/render.py` con `render_pagina_desde_fichero()`
- `data/04_eda/df_eda_final.parquet` — para verificar disponibilidad

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `docs/html/fase4/fase4_index.html` | Índice navegable de Fase 4 |

## 🔄 Flujo

```
render_pagina_desde_fichero() → HTML → escribe en disco
```

## ➡️ Siguiente

`f4_m01_inspeccion.ipynb` — inspección del dataset procesado


In [1]:
# CELDA 1: CONFIGURACION
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / "src").exists():
        break
    ROOT = ROOT.parent

    ROOT = Path.cwd()
    for _ in range(5):
        if (ROOT / "src").exists(): break
        ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
import numpy as np
from src.config import RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.html import (generar_kpis_html, generar_tarjetas_html,
                      generar_seccion_html, generar_html_navegacion_completa,
                      guardar_html)
from src.html.render import render_base_html
from src.utils.graficos import COLORES
RUTA_EDA = ROOT / "data" / "04_eda"
RUTA_FASE4_HTML = RUTA_HTML / "fase4"
crear_directorios([RUTA_FASE4_HTML])
fmt = formato_numero_es
info_entorno()


✓ Directorios verificados: 1
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# CELDA 2: CARGAR DATASET Y CALCULAR KPIs
df = pd.read_parquet(RUTA_EDA / "df_eda_final.parquet")
n_filas, n_cols = df.shape
n_features = n_cols - 2
pct_abandono = df["abandono"].mean() * 100
n_abandonan = int(df["abandono"].sum())
n_permanecen = n_filas - n_abandonan
n_categoricas = df.select_dtypes(include="object").shape[1]
n_numericas = df.select_dtypes(include="number").shape[1] - 1
missings = df.isnull().sum()
vars_con_nulos = missings[missings > 0].sort_values(ascending=False)
print("="*70)
print("DATASET: df_eda_final.parquet")
print("="*70)
print(f"Filas:      {fmt(n_filas)}")
print(f"Columnas:   {n_cols} ({n_features} features + titulacion + target)")
print(f"Abandono:   {fmt(n_abandonan)} ({pct_abandono:.1f}%)")
print(f"Permanecen: {fmt(n_permanecen)} ({100-pct_abandono:.1f}%)")
print(f"Numericas:  {n_numericas}")
print(f"Categoricas:{n_categoricas}")
print(f"\nVariables con nulos ({len(vars_con_nulos)}):")
for v, n in vars_con_nulos.items():
    print(f"  {v}: {fmt(n)} ({n/n_filas*100:.1f}%)")


DATASET: df_eda_final.parquet
Filas:      33.621
Columnas:   21 (19 features + titulacion + target)
Abandono:   9.833 (29.2%)
Permanecen: 23.788 (70.8%)
Numericas:  12
Categoricas:8

Variables con nulos (7):
  situacion_laboral: 12.740 (37.9%)
  orden_preferencia: 4.699 (14.0%)
  universidad_origen: 4.699 (14.0%)
  nota_selectividad: 3.118 (9.3%)
  nota_acceso: 2.567 (7.6%)
  nota_1er_anio: 2.353 (7.0%)
  max_pagos: 3 (0.0%)


In [3]:
# CELDA 3: GENERAR HTML
nav_fases, nav_modulos = generar_html_navegacion_completa(
    fase_activa='fase4', modulo_activo='indice'
)

# KPIs
KPIS = [
    {'valor': fmt(n_filas), 'titulo': 'Expedientes', 'color': COLORES['primary']},
    {'valor': str(n_features), 'titulo': 'Features', 'color': COLORES['success']},
    {'valor': f'{pct_abandono:.1f}%', 'titulo': 'Abandono', 'color': COLORES['danger']},
    {'valor': str(len(vars_con_nulos)), 'titulo': 'Vars con nulos', 'color': COLORES['warning']},
]

# Tarjetas — nombres y links actualizados con módulos reales
tarjetas = [
    {'titulo': 'M01: Inspección',
     'descripcion': f'Shape {fmt(n_filas)}×{n_cols}, dtypes, nulos, outliers. Visión general del dataset.',
     'emoji': '🔍', 'link': 'm01_inspeccion.html',
     'link_texto': 'Ver módulo', 'color': '#3182ce'},
    {'titulo': 'M02: Target',
     'descripcion': f'Abandono ({pct_abandono:.1f}%). Por rama, sexo, beca, vía acceso, titulación.',
     'emoji': '🎯', 'link': 'm02_target.html',
     'link_texto': 'Ver módulo', 'color': '#e53e3e'},
    {'titulo': 'M03: Distribuciones Numéricas',
     'descripcion': f'{n_numericas} variables numéricas. Histogramas, violines, QQ-plots, semáforo de escalado.',
     'emoji': '📈', 'link': 'm03_distribuciones_num.html',
     'link_texto': 'Ver módulo', 'color': '#38a169'},
    {'titulo': 'M04: Distribuciones Categóricas',
     'descripcion': f'{n_categoricas} variables categóricas. Chi², V Cramér, encoding recomendado, tasas abandono.',
     'emoji': '📊', 'link': 'm04_distribuciones_cat.html',
     'link_texto': 'Ver módulo', 'color': '#805ad5'},
    {'titulo': 'M05: Bivariante vs Target',
     'descripcion': 'Mann-Whitney + Cohen d para numéricas. Chi² + V Cramér para categóricas. Ranking global.',
     'emoji': '🔬', 'link': 'm05_bivariante.html',
     'link_texto': 'Ver módulo', 'color': '#d69e2e'},
    {'titulo': 'M06: Correlaciones',
     'descripcion': 'Heatmap Pearson, clustermap jerárquico, VIF, información mutua. Multicolinealidad.',
     'emoji': '🌡️', 'link': 'm06_correlaciones.html',
     'link_texto': 'Ver módulo', 'color': '#dd6b20'},
    {'titulo': 'M07: Selección de Features',
     'descripcion': 'Cruce de 3 fuentes: estadística + AutoML + VIF. Lista definitiva para Fase 5.',
     'emoji': '🎯', 'link': 'm07_seleccion_features.html',
     'link_texto': 'Ver módulo', 'color': '#319795'},
    {'titulo': 'M08: Comparativa Grupos',
     'descripcion': 'Radar, violín, KDE, semáforo Cohen d. Abandono vs éxito en todas las features.',
     'emoji': '⚡', 'link': 'm08_perfiles_riesgo.html',
     'link_texto': 'Ver módulo', 'color': '#2d3748'},
    {'titulo': 'M08b: Reducción Dimensional',
     'descripcion': 'PCA + t-SNE + UMAP. Separabilidad de grupos en espacio de baja dimensión.',
     'emoji': '🔭', 'link': 'm08b_reduccion_dimensional.html',
     'link_texto': 'Ver módulo', 'color': '#6b46c1'},
    {'titulo': 'M09: Conclusiones EDA',
     'descripcion': 'Ranking consolidado, semáforo Fase 5, análisis titulación, checklist entrada modelado.',
     'emoji': '📋', 'link': 'm09_conclusiones_eda.html',
     'link_texto': 'Ver módulo', 'color': '#1a365d'},
]

tarjetas_html = generar_tarjetas_html(tarjetas)
seccion_modulos = generar_seccion_html('Modulos de Análisis', tarjetas_html)

# Contexto AutoML
contexto_html = (
    '<div style="background:#f0f4f8;padding:20px;border-radius:10px;'
    'border-left:4px solid #3182ce;">'
    '<p><strong>Modelo ganador:</strong> Stacking (CatBoost + Random Forest + LogReg meta-learner). '
    'AUC = 0.9308 · F1 = 0.7882 · Dataset D_strict ({fmt(n_filas)} × {n_features} features).</p>'
    '<p><strong>Objetivo Fase 4:</strong> Analizar las 24 features auditadas con estadística '
    'clásica para confirmar hallazgos del AutoML, documentar decisiones de preprocesamiento '
    'y preparar el pipeline de Fase 5.</p>'
    '</div>'
)
seccion_contexto = generar_seccion_html('Contexto del Proyecto', contexto_html)

# Tabla missings
if len(vars_con_nulos) > 0:
    filas_tabla = ''
    for v, n in vars_con_nulos.items():
        pct = n / n_filas * 100
        color_f = '#fed7d7' if pct > 30 else '#fefcbf' if pct > 10 else '#f0fff4'
        filas_tabla += (
            f'<tr style="background:{color_f};">'
            f'<td style="padding:8px;"><code>{v}</code></td>'
            f'<td style="text-align:right;padding:8px;">{fmt(n)}</td>'
            f'<td style="text-align:right;padding:8px;">{pct:.1f}%</td></tr>'
        )
    tabla_nulos = (
        '<table style="width:100%;border-collapse:collapse;">'
        '<tr style="background:#3182ce;color:white;">'
        '<th style="padding:10px;text-align:left;">Variable</th>'
        '<th style="text-align:right;padding:10px;">Nulos</th>'
        '<th style="text-align:right;padding:10px;">%</th></tr>'
        + filas_tabla + '</table>'
    )
    seccion_nulos = generar_seccion_html('Variables con Valores Faltantes', tabla_nulos)
else:
    seccion_nulos = ''

# Ensamblar
contenido = generar_kpis_html(KPIS) + seccion_contexto + seccion_modulos + seccion_nulos

html = render_base_html(
    titulo='Fase 4: EDA Final',
    icono=chr(0x1f52c),
    subtitulo='Análisis Exploratorio del Dataset Procesado | TFM Abandono Universitario',
    nav_fases=nav_fases,
    nav_modulos=nav_modulos,
    contenido=contenido,
    notebook_nombre='f4_m00_indice.ipynb',
    notebook_carpeta='fase4_eda'
)

ruta_html = RUTA_FASE4_HTML / 'fase4_index.html'
guardar_html(html, ruta_html)
print(f'HTML generado: {ruta_html}')
print(f'Contiene {len(tarjetas)} tarjetas')


✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase4\fase4_index.html
HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase4\fase4_index.html
Contiene 8 tarjetas


In [4]:
# CELDA 4: RESUMEN
print('='*70)
print('F4-M00: INDICE - RESUMEN')
print('='*70)
print(f'Dataset:    df_eda_final.parquet')
print(f'Filas:      {fmt(n_filas)}')
print(f'Columnas:   {n_cols} ({n_features} features + titulacion + target)')
print(f'Abandono:   {pct_abandono:.1f}%')
print(f'Missings:   {len(vars_con_nulos)} variables con nulos')
print(f'HTML:       {ruta_html}')
print(f'Módulos:    {len(tarjetas)} (M01-M09 + M08b)')
print()
print('Siguiente: f4_m01_inspeccion.ipynb')


F4-M00: INDICE - RESUMEN
Dataset:    df_eda_final.parquet
Filas:      33.621
Columnas:   21 (19 features + titulacion + target)
Abandono:   29.2%
Missings:   7 variables con nulos
HTML:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase4\fase4_index.html
Modulos:    8

Siguiente: f4_m01_inspeccion.ipynb
